# 🚨 AOG 자재 수급 어시스턴트

채팅창에 상황을 말하면 AI가 조건을 읽어 즉시 1~6단계를 자동으로 훑고, 실제 연락/메일 발송이 필요한 지점에서만 승인을 구하는 Streamlit 앱입니다. 아래 3개 셀을 순서대로 실행하세요. 사용법·테스트 시나리오는 저장소의 `README.md`를 참고하세요.

## Cell 1 — 라이브러리 설치

**이전 버전과 달라진 점**: `--upgrade`와 강제 재시작(`os.kill`)을 없앴습니다. 두 가지 모두 이미 만족하는 Colab 기본 패키지(numpy/pandas 등)를 불필요하게 건드려 오히려 충돌을 만들 수 있는 요인이었습니다. 지금은 **어떤 사전 설치 패키지도 건드리지 않고, 아직 없는 패키지만 그대로 설치**합니다 — 설치 후 런타임을 재시작할 필요가 없습니다. 바로 Cell 2로 넘어가세요.

- `-q`(quiet)를 빼서 pip 출력이 전부 보이도록 했습니다 — 만약 설치가 실패하면, 이 셀의 출력(빨간 글씨 오류 메시지) 전체를 그대로 복사해서 알려주세요. 그래야 정확한 원인을 잡을 수 있습니다.
- `streamlit`, `plotly`는 필수, `transformers`는 선택(4·5단계 긴급 메일을 AI가 다듬는 기능에만 사용) 패키지라 설치 명령을 분리했습니다 — `transformers` 설치가 느리거나 실패해도 앱의 핵심 기능은 전혀 영향받지 않습니다.

> 그래도 위젯마다 `TypeError`가 난다면: Colab 메뉴에서 **런타임 → 세션 다시 시작**을 누른 뒤 Cell 1은 건너뛰고 Cell 2부터 다시 실행해보세요. (설치 자체는 이미 되어 있으므로 Cell 1을 다시 실행할 필요는 없습니다.)

In [ ]:
!pip install streamlit plotly
!pip install transformers


## Cell 2 — Streamlit 앱 코드 작성 (`app.py`)

In [ ]:
%%writefile app.py
# ============================================================================
# AOG(Aircraft on Ground) 자재 수급 어시스턴트 — Streamlit (7단계 · 최적화/자동화)
# ----------------------------------------------------------------------------
# 7단계 절차(모두 데이터 기반):
#   1 FAK 키트(기종별 소모품 패키지, 기체 탑재 → 공항 무관)
#   2 로컬 Allocation(발생 공항 자사 창고)
#   3 이송 최적화(타 공항 자사 창고 + 제휴 창고 Non-Pool → Lead Time 최단 추천)
#   4 Pooling(파트너 재고 + coverage 공항 일치)
#   5 발생 공항 Main Station 타사 문의(기종 운영사 교집합, 영문 메일)
#   6 동일 기종 운영 타사 전체 문의(영문 메일)
#   7 Hand-carry/Cargo 파송(진에어 화물인가 검증 + AI 인보이스 + 야간 통관 가이드)
# 입력=기번. aircraft_registry가 기번→(기종, 운영사) 매핑. 타사 문의는 기종 기준.
# ============================================================================

import copy
import json
import os
import re
import urllib.parse
from collections import Counter
from datetime import datetime, timedelta

import pandas as pd
import plotly.graph_objects as go
import streamlit as st

try:
    import transformers  # noqa: F401
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False

st.set_page_config(page_title="AOG 자재 수급 어시스턴트", page_icon="🚨", layout="wide")

DB_PATH = "aog_db.json"
SCHEMA_VERSION = "9"

# 운영사별 카고 부킹 리드타임(대한항공 카고 사이트 편명 조회·부킹 소요 반영)
CARGO_BOOKING_HOURS = {"대한항공": 3.0, "진에어": 4.0}

AIRPORT_COORDS = {
    "ICN": (37.4602, 126.4407, "인천"), "GMP": (37.5583, 126.7906, "김포"),
    "NRT": (35.7647, 140.3864, "나리타"), "HND": (35.5494, 139.7798, "하네다"),
    "CDG": (49.0097, 2.5479, "파리 CDG"), "JFK": (40.6413, -73.7781, "뉴욕 JFK"),
    "SIN": (1.3644, 103.9915, "싱가포르"), "HKG": (22.3080, 113.9185, "홍콩"),
    "FRA": (50.0379, 8.5622, "프랑크푸르트"), "LAX": (33.9416, -118.4085, "LA"),
    "BKK": (13.6900, 100.7501, "방콕"), "SYD": (-33.9399, 151.1753, "시드니"),
}

# ----------------------------------------------------------------------------
# 1-a. 기번 등록부 (기번 -> 기종, 운영사)
# ----------------------------------------------------------------------------
_AIRCRAFT_REGISTRY = [
    {"registration": "HL7702", "aircraft_type": "A330-300", "operator": "대한항공"},
    {"registration": "HL7710", "aircraft_type": "A330-300", "operator": "대한항공"},
    {"registration": "HL7720", "aircraft_type": "A330-300", "operator": "대한항공"},
    {"registration": "HL8008", "aircraft_type": "B777-300ER", "operator": "대한항공"},
    {"registration": "HL8009", "aircraft_type": "B777-300ER", "operator": "대한항공"},
    {"registration": "HL8010", "aircraft_type": "B777-300ER", "operator": "대한항공"},
    {"registration": "HL8259", "aircraft_type": "B737-800", "operator": "대한항공"},
    {"registration": "HL8260", "aircraft_type": "B737-800", "operator": "대한항공"},
    {"registration": "HL8261", "aircraft_type": "B737-800", "operator": "대한항공"},
    {"registration": "HL8501", "aircraft_type": "A321neo", "operator": "대한항공"},
    {"registration": "HL8502", "aircraft_type": "A321neo", "operator": "대한항공"},
    {"registration": "HL8081", "aircraft_type": "B787-9", "operator": "대한항공"},
    {"registration": "HL8082", "aircraft_type": "B787-9", "operator": "대한항공"},
    {"registration": "HL8084", "aircraft_type": "B787-9", "operator": "대한항공"},
    {"registration": "HL8360", "aircraft_type": "A350-900", "operator": "대한항공"},
    {"registration": "HL8361", "aircraft_type": "A350-900", "operator": "대한항공"},
    # 진에어(LCC 자회사) — AOG 프로세스가 다름(화물 인가 있는 공항으로만 자사 카고 파송 가능)
    {"registration": "LJ2201", "aircraft_type": "B737-800", "operator": "진에어"},
    {"registration": "LJ2202", "aircraft_type": "B737-800", "operator": "진에어"},
    {"registration": "LJ7771", "aircraft_type": "B777-300ER", "operator": "진에어"},
]

# ----------------------------------------------------------------------------
# 1-b. FAK 키트(기종별 소모품 패키지) — 반드시 소모품/비상품목만. (로터블은 넣지 않음)
# ----------------------------------------------------------------------------
_FAK_CONTENTS = {
    "A330-300": [("OXY-GEN-A330-15", "Chemical Oxygen Generator", 2), ("SMK-DET-A330-21", "Cargo Compartment Smoke Detector", 3),
                 ("LIFE-VEST-A330-40", "Passenger Life Vest", 8), ("FIRE-EXT-A330-33", "Portable Halon Fire Extinguisher", 4),
                 ("O2-BOTTLE-A330-27", "Portable Oxygen Bottle", 5), ("ELT-A330-51", "Emergency Locator Transmitter", 1)],
    "B777-300ER": [("OXY-GEN-777-15", "Chemical Oxygen Generator", 2), ("SMK-DET-777-22", "Cargo Compartment Smoke Detector", 3),
                   ("LIFE-VEST-777-40", "Passenger Life Vest", 10), ("FIRE-EXT-777-33", "Portable Halon Fire Extinguisher", 5),
                   ("O2-BOTTLE-777-27", "Portable Oxygen Bottle", 6), ("ELT-777-51", "Emergency Locator Transmitter", 1)],
    "B737-800": [("OXY-GEN-737-15", "Chemical Oxygen Generator", 2), ("SMK-DET-737-04", "Cargo Compartment Smoke Detector", 3),
                 ("LIFE-VEST-737-42", "Passenger Life Vest", 8), ("FIRE-EXT-737-33", "Portable Halon Fire Extinguisher", 3),
                 ("O2-BOTTLE-737-08", "Portable Oxygen Bottle", 5), ("ELT-737-51", "Emergency Locator Transmitter", 1)],
    "A321neo": [("OXY-GEN-321N-15", "Chemical Oxygen Generator", 2), ("SMK-DET-321N-43", "Cargo Compartment Smoke Detector", 3),
                ("LIFE-VEST-321N-17", "Passenger Life Vest", 8), ("FIRE-EXT-321N-33", "Portable Halon Fire Extinguisher", 3),
                ("CAB-PRESS-VLV-321N-06", "Cabin Pressure Relief Valve", 2), ("ELT-321N-51", "Emergency Locator Transmitter", 1)],
    "B787-9": [("OXY-GEN-787-15", "Chemical Oxygen Generator", 2), ("SMK-DET-787-44", "Cargo Compartment Smoke Detector", 3),
               ("LIFE-VEST-787-40", "Passenger Life Vest", 10), ("FIRE-EXT-787-14", "Portable Halon Fire Extinguisher", 4),
               ("EVAC-SLIDE-787-02", "Emergency Evacuation Slide", 1), ("ELT-787-51", "Emergency Locator Transmitter", 1)],
    "A350-900": [("OXY-GEN-350-19", "Chemical Oxygen Generator", 2), ("SMOKE-DET-350-11", "Cargo Compartment Smoke Detector", 2),
                 ("LIFE-VEST-350-45", "Passenger Life Vest", 10), ("FIRE-EXT-350-33", "Portable Halon Fire Extinguisher", 4),
                 ("O2-BOTTLE-350-27", "Portable Oxygen Bottle", 6), ("ELT-350-51", "Emergency Locator Transmitter", 1)],
}
_FAK_KITS = [{"aircraft_type": t, "kit_name": f"{t} 표준 FAK 패키지",
              "contents": [{"part_number": pn, "part_name": nm, "qty": q} for pn, nm, q in items]}
             for t, items in _FAK_CONTENTS.items()]

# ----------------------------------------------------------------------------
# 1-c. 로터블 마스터(기종별 파트넘버->이름). Allocation/제휴/Pooling 이름 조회에 사용.
# ----------------------------------------------------------------------------
_ROTABLE_MASTER = {
    "A330-300": {"IDG-A330-001": "Integrated Drive Generator", "FUEL-NOZ-A330-07": "Engine Fuel Nozzle",
                 "BRK-A330-CARBON-12": "Carbon Brake Assembly", "HYD-PUMP-A330-18": "Engine-Driven Hydraulic Pump",
                 "WHL-MLG-A330-22": "Main Landing Gear Wheel", "STARTER-GEN-A330-24": "Starter Generator",
                 "APU-A330-30": "Auxiliary Power Unit", "FMS-A330-40": "Flight Management Computer"},
    "B777-300ER": {"BRK-B777-CARBON-01": "Carbon Brake Assembly", "HYD-PUMP-777-23": "Engine-Driven Hydraulic Pump",
                   "WHL-MLG-777-06": "Main Landing Gear Wheel", "IDG-777-11": "Integrated Drive Generator",
                   "ENG-FAN-BLADE-777-31": "Engine Fan Blade", "APU-777-30": "Auxiliary Power Unit",
                   "ADIRU-777-35": "Air Data Inertial Reference Unit", "FMS-777-40": "Flight Management Computer"},
    "B737-800": {"STARTER-GEN-737-03": "Starter Generator", "WHL-MLG-737-25": "Main Landing Gear Wheel",
                 "HYD-PUMP-737-11": "Engine-Driven Hydraulic Pump", "BRK-737-CARBON-14": "Carbon Brake Assembly",
                 "AVIONICS-FMS-737-32": "Flight Management Computer", "APU-737-30": "Auxiliary Power Unit"},
    "A321neo": {"FLAP-TRK-ROLLER-321N-08": "Flap Track Roller", "BRK-321N-CARBON-26": "Carbon Brake Assembly",
                "APU-321N-30": "Auxiliary Power Unit", "AVIONICS-FMS-321N-04": "Flight Management Computer",
                "WHL-NLG-321N-22": "Nose Landing Gear Wheel", "IDG-321N-11": "Integrated Drive Generator"},
    "B787-9": {"APU-787-01": "Auxiliary Power Unit", "IDG-787-27": "Integrated Drive Generator",
               "CARGO-DOOR-ACT-787-03": "Cargo Door Actuator", "BRK-787-CARBON-12": "Carbon Brake Assembly",
               "WHL-MLG-787-22": "Main Landing Gear Wheel", "FMS-787-40": "Flight Management Computer"},
    "A350-900": {"IDG-A350-002": "Integrated Drive Generator", "FUEL-PUMP-350-28": "Fuel Boost Pump",
                 "BRK-350-CARBON-29": "Carbon Brake Assembly", "WHL-MLG-350-22": "Main Landing Gear Wheel",
                 "APU-350-30": "Auxiliary Power Unit", "FMS-350-40": "Flight Management Computer"},
}
_ROTABLE_NAME = {pn: nm for parts in _ROTABLE_MASTER.values() for pn, nm in parts.items()}

# 스테이션별 자사 창고 보유 로터블 (airport, type, pn, qty) — ICN 허브 최다, 해외 소량.
_STATION_STOCK = [
    ("ICN", "A330-300", "IDG-A330-001", 2), ("ICN", "A330-300", "FUEL-NOZ-A330-07", 4),
    ("ICN", "A330-300", "STARTER-GEN-A330-24", 1), ("ICN", "A330-300", "FMS-A330-40", 1),
    ("ICN", "B777-300ER", "BRK-B777-CARBON-01", 1), ("ICN", "B777-300ER", "HYD-PUMP-777-23", 2),
    ("ICN", "B777-300ER", "APU-777-30", 1), ("ICN", "B737-800", "STARTER-GEN-737-03", 2),
    ("ICN", "B737-800", "BRK-737-CARBON-14", 2), ("ICN", "A321neo", "FLAP-TRK-ROLLER-321N-08", 5),
    ("ICN", "A321neo", "BRK-321N-CARBON-26", 2), ("ICN", "B787-9", "APU-787-01", 1),
    ("ICN", "B787-9", "IDG-787-27", 1), ("ICN", "A350-900", "IDG-A350-002", 1),
    ("ICN", "A350-900", "FUEL-PUMP-350-28", 2),
    ("GMP", "B737-800", "WHL-MLG-737-25", 3), ("GMP", "A321neo", "WHL-NLG-321N-22", 2),
    ("NRT", "B787-9", "IDG-787-27", 1), ("NRT", "A330-300", "STARTER-GEN-A330-24", 1),
    ("NRT", "B777-300ER", "APU-777-30", 1),
    ("CDG", "A330-300", "FMS-A330-40", 1), ("CDG", "B787-9", "CARGO-DOOR-ACT-787-03", 1),
    ("FRA", "A330-300", "IDG-A330-001", 1), ("FRA", "B777-300ER", "ENG-FAN-BLADE-777-31", 1),
    ("LAX", "B777-300ER", "BRK-B777-CARBON-01", 1), ("LAX", "B787-9", "BRK-787-CARBON-12", 1),
    ("JFK", "B787-9", "WHL-MLG-787-22", 2), ("JFK", "B777-300ER", "WHL-MLG-777-06", 1),
    ("HKG", "B777-300ER", "HYD-PUMP-777-23", 1), ("HKG", "A350-900", "BRK-350-CARBON-29", 1),
    ("SIN", "A350-900", "FUEL-PUMP-350-28", 1), ("SIN", "A330-300", "STARTER-GEN-A330-24", 1),
    ("BKK", "A330-300", "BRK-A330-CARBON-12", 1), ("SYD", "A350-900", "WHL-MLG-350-22", 1),
]
_WAREHOUSE_NAMES = {
    "ICN": "ICN 본사 통합 자재창고", "GMP": "GMP 지점 자재창고", "NRT": "NRT 해외지점 창고",
    "HND": "HND 해외지점 창고", "CDG": "CDG 해외지점 창고", "JFK": "JFK 해외지점 창고",
    "SIN": "SIN 해외지점 창고", "HKG": "HKG 해외지점 창고", "FRA": "FRA 해외지점 창고",
    "LAX": "LAX 해외지점 창고", "BKK": "BKK 해외지점 창고", "SYD": "SYD 해외지점 창고",
}


def _build_station_warehouses():
    by_ap = {}
    for ap, t, pn, qty in _STATION_STOCK:
        by_ap.setdefault(ap, []).append(
            {"aircraft_type": t, "part_number": pn, "part_name": _ROTABLE_NAME.get(pn, pn), "qty": qty})
    return [{"airport": ap, "warehouse_name": _WAREHOUSE_NAMES.get(ap, f"{ap} 자재창고"), "contents": items}
            for ap, items in by_ap.items()]


# ----------------------------------------------------------------------------
# 1-d. 제휴 창고(Non-Pool) — 정식 풀 계약은 없으나 제휴로 땡겨올 수 있는 외부 창고.
# ----------------------------------------------------------------------------
def _rot(pn):
    return _ROTABLE_NAME.get(pn, pn)


_ALLIANCE_WAREHOUSES = [
    {"airport": "CDG", "partner": "Air France Industries (제휴)",
     "contents": [{"aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "part_name": _rot("FUEL-NOZ-A330-07"), "qty": 1}]},
    {"airport": "JFK", "partner": "Delta TechOps (제휴)",
     "contents": [{"aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "part_name": _rot("HYD-PUMP-737-11"), "qty": 1}]},
    {"airport": "FRA", "partner": "Lufthansa Technik (제휴)",
     "contents": [{"aircraft_type": "B777-300ER", "part_number": "BRK-B777-CARBON-01", "part_name": _rot("BRK-B777-CARBON-01"), "qty": 1}]},
    {"airport": "SIN", "partner": "SIA Engineering (제휴)",
     "contents": [{"aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "part_name": _rot("HYD-PUMP-737-11"), "qty": 2}]},
    {"airport": "LAX", "partner": "Menzies Aviation (제휴)",
     "contents": [{"aircraft_type": "B787-9", "part_number": "BRK-787-CARBON-12", "part_name": _rot("BRK-787-CARBON-12"), "qty": 1}]},
]

# ----------------------------------------------------------------------------
# 1-e. Pooling 파트너/재고
# ----------------------------------------------------------------------------
_POOLING_STOCK_RAW = [
    ("SIA Engineering (싱가포르)", "A350-900", "BRK-350-CARBON-29", 1, "SIN 창고"),
    ("Lufthansa Technik (독일)", "A330-300", "IDG-A330-001", 1, "FRA 창고"),
    ("Lufthansa Technik (독일)", "A321neo", "APU-321N-30", 1, "FRA 창고"),
    ("HAECO (홍콩)", "B777-300ER", "WHL-MLG-777-06", 2, "HKG 창고"),
    ("AAR Corp (미국)", "A321neo", "AVIONICS-FMS-321N-04", 1, "JFK 창고"),
    ("AAR Corp (미국)", "B787-9", "WHL-MLG-787-22", 1, "JFK 창고"),
    ("ANA Base Maintenance (일본)", "B787-9", "CARGO-DOOR-ACT-787-03", 1, "NRT 창고"),
    ("MTU Maintenance (독일)", "B777-300ER", "ENG-FAN-BLADE-777-31", 1, "FRA 창고"),
    ("Delta TechOps (미국)", "B737-800", "AVIONICS-FMS-737-32", 1, "JFK 창고"),
]
_POOLING_STOCK = [{"partner": p, "aircraft_type": t, "part_number": pn, "part_name": _rot(pn), "qty": q, "location": loc}
                  for p, t, pn, q, loc in _POOLING_STOCK_RAW]

# ----------------------------------------------------------------------------
# 1-f. 스테이션 정보(조업사/주소/진에어 화물인가) — 인보이스 자동화 + 진에어 Auth
# ----------------------------------------------------------------------------
_STATION_INFO = [
    {"airport": "ICN", "jinair_cargo_auth": "Y", "handling_agent": "KAS (Korean Air Service)", "address": "285, Jega-ro, Jung-gu, Incheon, Republic of Korea"},
    {"airport": "GMP", "jinair_cargo_auth": "Y", "handling_agent": "KAS Gimpo", "address": "Gimpo Int'l Airport Cargo Terminal, Gangseo-gu, Seoul, Republic of Korea"},
    {"airport": "NRT", "jinair_cargo_auth": "Y", "handling_agent": "ANA Cargo", "address": "Narita International Airport, Furugome, Narita, Chiba, Japan"},
    {"airport": "HND", "jinair_cargo_auth": "Y", "handling_agent": "JAL Cargo", "address": "Haneda Airport Cargo Terminal, Ota-ku, Tokyo, Japan"},
    {"airport": "JFK", "jinair_cargo_auth": "N", "handling_agent": "Delta Cargo", "address": "JFK Int'l Airport, Cargo Bldg 75, Jamaica, NY 11430, USA"},
    {"airport": "CDG", "jinair_cargo_auth": "N", "handling_agent": "Air France Cargo", "address": "Roissy CDG, 9 Route de l'Arpenteur, Tremblay-en-France, France"},
    {"airport": "FRA", "jinair_cargo_auth": "N", "handling_agent": "Fraport Cargo", "address": "Frankfurt Airport CargoCity Süd, 60549 Frankfurt, Germany"},
    {"airport": "LAX", "jinair_cargo_auth": "Y", "handling_agent": "Mercury Air Cargo", "address": "Los Angeles Int'l Airport, 6040 Avion Dr, Los Angeles, CA 90045, USA"},
    {"airport": "SIN", "jinair_cargo_auth": "Y", "handling_agent": "SATS Cargo", "address": "Singapore Changi Airport, Airfreight Terminal 1, Singapore"},
    {"airport": "HKG", "jinair_cargo_auth": "Y", "handling_agent": "HACTL", "address": "Hong Kong Int'l Airport, SuperTerminal 1, Chek Lap Kok, Hong Kong"},
    {"airport": "BKK", "jinair_cargo_auth": "Y", "handling_agent": "BFS Cargo", "address": "Suvarnabhumi Airport Free Zone, Samut Prakan, Thailand"},
    {"airport": "SYD", "jinair_cargo_auth": "N", "handling_agent": "Menzies Aviation SYD", "address": "Sydney Kingsford Smith Airport, Cargo Precinct, NSW 2020, Australia"},
]

# ----------------------------------------------------------------------------
# 1-g. 야간 통관 가이드(관세사 부재 시 담당자가 직접 수행) — 휴먼에러 방지 체크리스트
# ----------------------------------------------------------------------------
_NIGHT_CUSTOMS_MANUAL = [
    "1. [신고 누락 방지] 관세청 유니패스(UNI-PASS) 접속 → 임시 수출/재수입 신고서 사전 작성",
    "2. AOG Declaration(증빙) + 상용 인보이스 사본 필수 첨부하여 전송",
    "3. 야간 당직 관세사(010-XXXX-XXXX) 유선 연락 → AOG 긴급 통관 승인 요청",
    "4. 승인 번호(MRN) 획득 확인 후 조업사 인계 (미확인 인계 시 휴먼에러로 파송 지연!)",
    "5. 도착지 세관 야간창구 사전 통지 (조업사 주소·수취인 정보 재확인 → 오배송 방지)",
]

DEFAULT_DB = {
    "_schema_version": SCHEMA_VERSION,
    "aircraft_registry": _AIRCRAFT_REGISTRY,
    "fak_kits": _FAK_KITS,
    "station_warehouses": _build_station_warehouses(),
    "alliance_warehouses": _ALLIANCE_WAREHOUSES,
    "pooling_partners": [
        {"partner": "SIA Engineering (싱가포르)", "contact": "+65-6541-2000", "email": "pooling@siae.com.sg", "coverage_airports": "SIN,HKG,BKK,NRT,ICN,SYD"},
        {"partner": "Lufthansa Technik (독일)", "contact": "+49-40-5070-5553", "email": "pooling@lht.dlh.de", "coverage_airports": "FRA,CDG"},
        {"partner": "ANA Base Maintenance (일본)", "contact": "+81-3-6735-1111", "email": "pooling@ana.co.jp", "coverage_airports": "NRT,HND,ICN"},
        {"partner": "HAECO (홍콩)", "contact": "+852-2767-6111", "email": "pooling@haeco.com", "coverage_airports": "HKG,SIN,BKK"},
        {"partner": "AAR Corp (미국)", "contact": "+1-630-227-2000", "email": "pooling@aarcorp.com", "coverage_airports": "JFK,LAX"},
        {"partner": "MTU Maintenance (독일)", "contact": "+49-511-7807-0", "email": "pooling@mtu.de", "coverage_airports": "FRA,CDG"},
        {"partner": "Delta TechOps (미국)", "contact": "+1-404-715-2600", "email": "pooling@delta.com", "coverage_airports": "JFK,LAX,ICN"},
    ],
    "pooling_stock": _POOLING_STOCK,
    "main_station_airlines": [
        {"airport": "ICN", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "ops.icn@flyasiana.com"},
        {"airport": "GMP", "airline": "제주항공", "contact": "02-2015-1000", "email": "ops.gmp@jejuair.net"},
        {"airport": "NRT", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "ops.nrt@ana.co.jp"},
        {"airport": "HND", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "ops.hnd@jal.com"},
        {"airport": "CDG", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "ops.cdg@airfrance.fr"},
        {"airport": "JFK", "airline": "Delta Air Lines", "contact": "+1-800-221-1212", "email": "ops.jfk@delta.com"},
        {"airport": "SIN", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "ops.sin@singaporeair.com"},
        {"airport": "HKG", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "ops.hkg@cathaypacific.com"},
        {"airport": "FRA", "airline": "Lufthansa", "contact": "+49-69-86799799", "email": "ops.fra@lufthansa.com"},
        {"airport": "LAX", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "ops.lax@united.com"},
        {"airport": "BKK", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "ops.bkk@thaiairways.com"},
        {"airport": "SYD", "airline": "Qantas", "contact": "+61-2-9691-3636", "email": "ops.syd@qantas.com.au"},
    ],
    "fleet_operators": [
        {"aircraft_type": "A330-300", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a330@flyasiana.com"},
        {"aircraft_type": "A330-300", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a330@cathaypacific.com"},
        {"aircraft_type": "A330-300", "airline": "China Eastern", "contact": "+86-21-95530", "email": "fleet.a330@ceair.com"},
        {"aircraft_type": "A330-300", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "fleet.a330@thaiairways.com"},
        {"aircraft_type": "B777-300ER", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.b777@flyasiana.com"},
        {"aircraft_type": "B777-300ER", "airline": "Emirates", "contact": "+971-600-555555", "email": "fleet.b777@emirates.com"},
        {"aircraft_type": "B777-300ER", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b777@ana.co.jp"},
        {"aircraft_type": "B777-300ER", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.b777@cathaypacific.com"},
        {"aircraft_type": "B737-800", "airline": "제주항공", "contact": "02-2015-1000", "email": "fleet.b737@jejuair.net"},
        {"aircraft_type": "B737-800", "airline": "티웨이항공", "contact": "1688-8686", "email": "fleet.b737@twayair.com"},
        {"aircraft_type": "B737-800", "airline": "Ryanair", "contact": "+353-1-945-1212", "email": "fleet.b737@ryanair.com"},
        {"aircraft_type": "B737-800", "airline": "Qantas", "contact": "+61-2-9691-3636", "email": "fleet.b737@qantas.com.au"},
        {"aircraft_type": "A321neo", "airline": "에어부산", "contact": "1666-3060", "email": "fleet.a321n@airbusan.com"},
        {"aircraft_type": "A321neo", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "fleet.a321n@jal.com"},
        {"aircraft_type": "A321neo", "airline": "Wizz Air", "contact": "+36-1-777-9300", "email": "fleet.a321n@wizzair.com"},
        {"aircraft_type": "B787-9", "airline": "ANA", "contact": "+81-3-6735-1000", "email": "fleet.b787@ana.co.jp"},
        {"aircraft_type": "B787-9", "airline": "United Airlines", "contact": "+1-800-864-8331", "email": "fleet.b787@united.com"},
        {"aircraft_type": "B787-9", "airline": "JAL", "contact": "+81-3-5460-3121", "email": "fleet.b787@jal.com"},
        {"aircraft_type": "B787-9", "airline": "Air France", "contact": "+33-1-4356-7890", "email": "fleet.b787@airfrance.fr"},
        {"aircraft_type": "A350-900", "airline": "아시아나항공", "contact": "02-2669-8000", "email": "fleet.a350@flyasiana.com"},
        {"aircraft_type": "A350-900", "airline": "Cathay Pacific", "contact": "+852-2747-1888", "email": "fleet.a350@cathaypacific.com"},
        {"aircraft_type": "A350-900", "airline": "Qatar Airways", "contact": "+974-4023-0000", "email": "fleet.a350@qatarairways.com.qa"},
        {"aircraft_type": "A350-900", "airline": "Singapore Airlines", "contact": "+65-6223-8888", "email": "fleet.a350@singaporeair.com"},
        {"aircraft_type": "A350-900", "airline": "Thai Airways", "contact": "+66-2-356-1111", "email": "fleet.a350@thaiairways.com"},
    ],
    "station_info": _STATION_INFO,
    "night_customs_manual": _NIGHT_CUSTOMS_MANUAL,
    "allocation_dept_contacts": [
        {"airport": "ICN", "department": "자재관리팀 Allocation 파트", "contact": "02-XXXX-1000", "email": "allocation.icn@airline.example"},
        {"airport": "GMP", "department": "자재관리팀 Allocation 파트(김포)", "contact": "02-XXXX-1005", "email": "allocation.gmp@airline.example"},
        {"airport": "NRT", "department": "해외지점 Allocation(나리타)", "contact": "+81-3-XXXX-1010", "email": "allocation.nrt@airline.example"},
        {"airport": "CDG", "department": "해외지점 Allocation(파리)", "contact": "+33-1-XXXX-1015", "email": "allocation.cdg@airline.example"},
        {"airport": "HKG", "department": "해외지점 Allocation(홍콩)", "contact": "+852-XXXX-1020", "email": "allocation.hkg@airline.example"},
        {"airport": "SIN", "department": "해외지점 Allocation(싱가포르)", "contact": "+65-XXXX-1025", "email": "allocation.sin@airline.example"},
        {"airport": "FRA", "department": "해외지점 Allocation(프랑크푸르트)", "contact": "+49-69-XXXX-1030", "email": "allocation.fra@airline.example"},
        {"airport": "LAX", "department": "해외지점 Allocation(LA)", "contact": "+1-310-XXXX-1035", "email": "allocation.lax@airline.example"},
    ],
    "customs_team": {"name": "통관팀", "contact": "02-XXXX-2000", "email": "customs@airline.example"},
    "sourcing_history": [
        {"date": "2026-06-25", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "airport": "CDG", "resolved_step": "1·FAK", "source": "기체 탑재 FAK 키트", "method": "자체재고(FAK)", "result": "성공", "lead_time_hours": 1},
        {"date": "2026-05-12", "registration": "HL7710", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "airport": "FRA", "resolved_step": "4·Pooling", "source": "Lufthansa Technik (FRA)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 6},
        {"date": "2026-02-20", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "airport": "ICN", "resolved_step": "2·Allocation", "source": "ICN 본사 통합 자재창고", "method": "자체재고(Allocation)", "result": "성공", "lead_time_hours": 2},
        {"date": "2026-06-01", "registration": "HL7720", "aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "airport": "BKK", "resolved_step": "3·이송최적화", "source": "ICN 자사창고→BKK 이송", "method": "이송(자사)", "result": "성공", "lead_time_hours": 11},
        {"date": "2026-03-15", "registration": "HL8259", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "airport": "SIN", "resolved_step": "3·이송최적화", "source": "SIA Engineering 제휴창고(SIN)", "method": "제휴(Non-Pool)", "result": "성공", "lead_time_hours": 4},
        {"date": "2026-01-09", "registration": "HL8260", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "airport": "JFK", "resolved_step": "3·이송최적화", "source": "Delta TechOps 제휴창고(JFK)", "method": "제휴(Non-Pool)", "result": "성공", "lead_time_hours": 3},
        {"date": "2026-06-18", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "airport": "LAX", "resolved_step": "6·타 항공사", "source": "United Airlines (LAX)", "method": "대여(Loan)", "result": "실패", "lead_time_hours": 6},
        {"date": "2026-06-19", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "airport": "LAX", "resolved_step": "7·Hand-carry", "source": "본사 ICN→LAX KE011", "method": "Hand-carry", "result": "성공", "lead_time_hours": 30},
        {"date": "2026-07-02", "registration": "HL8084", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "airport": "NRT", "resolved_step": "4·Pooling", "source": "ANA Base Maintenance (NRT)", "method": "대여(Loan)", "result": "성공", "lead_time_hours": 5},
    ],
    "defect_history": [
        {"date": "2026-06-25", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "OXY-GEN-A330-15", "defect": "Oxygen generator past expiry", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "표준 공구", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-05-12", "registration": "HL7710", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "defect": "IDG low oil pressure warning", "resolution": "부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "IDG 인출 지그 JIG-IDG-01, 토크렌치", "result": "정상 복구", "downtime_hours": 6},
        {"date": "2026-02-20", "registration": "HL7702", "aircraft_type": "A330-300", "part_number": "IDG-A330-001", "defect": "IDG disconnect fault (intermittent)", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "BITE 테스터", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-06-01", "registration": "HL7720", "aircraft_type": "A330-300", "part_number": "FUEL-NOZ-A330-07", "defect": "Fuel nozzle coking / uneven spray", "resolution": "부품 교환", "parts_scope": "단일 부품", "tools_required": "노즐 풀러 PULLER-07", "result": "정상 복구", "downtime_hours": 3},
        {"date": "2026-03-15", "registration": "HL8259", "aircraft_type": "B737-800", "part_number": "HYD-PUMP-737-11", "defect": "Hydraulic pump low pressure", "resolution": "부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "유압 실링 키트, 토크렌치", "result": "정상 복구", "downtime_hours": 8},
        {"date": "2026-05-05", "registration": "HL8260", "aircraft_type": "B737-800", "part_number": "SMK-DET-737-04", "defect": "Cargo smoke detector false warning", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "표준 공구", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2026-06-18", "registration": "HL8082", "aircraft_type": "B787-9", "part_number": "APU-787-01", "defect": "APU start fault", "resolution": "디퍼(MEL) 후 부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "APU 호이스트 HOIST-APU-01", "result": "재발 → 부품 교환", "downtime_hours": 30},
        {"date": "2026-07-02", "registration": "HL8084", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "defect": "Cargo door actuator fail to latch", "resolution": "리셋/재시동", "parts_scope": "교환 없음(리셋)", "tools_required": "BITE 테스터", "result": "정상 복구", "downtime_hours": 1},
        {"date": "2025-12-11", "registration": "HL8081", "aircraft_type": "B787-9", "part_number": "CARGO-DOOR-ACT-787-03", "defect": "Cargo door actuator fail to latch", "resolution": "SW 리로드 후 부품 교환", "parts_scope": "패키지(어셈블리)", "tools_required": "액추에이터 리깅 툴 RIG-DR-03", "result": "정상 복구", "downtime_hours": 7},
    ],
}

STEP_NAMES = {
    1: "1단계 · FAK 키트 확인",
    2: "2단계 · 로컬 Allocation(발생 공항 창고) 확인",
    3: "3단계 · 이송 최적화(타 스테이션·제휴창고)",
    4: "4단계 · Pooling 파트너사 확인",
    5: "5단계 · 발생 공항 Main Station 타사 문의",
    6: "6단계 · 동일 기종 운영 타사 문의",
    7: "7단계 · Hand-carry/Cargo 파송",
}
TOTAL_STEPS = 7

DEFAULT_EMAIL_SUBJECT = "[AOG - URGENT] Spare part support request - {aircraft_type} / {part_number} / {airport}"
DEFAULT_EMAIL_BODY_TEMPLATE = (
    "Dear Operations / Material Support Team,\n\n"
    "We are currently experiencing an AOG (Aircraft on Ground) situation on our {aircraft_type} "
    "aircraft at {airport}. We urgently require the following part and would greatly appreciate your support:\n\n"
    "  - Part Number: {part_number}\n  - Aircraft Type: {aircraft_type}\n  - Station: {airport}\n\n"
    "If you have this part available, please advise on loan availability and the minimum lead time "
    "at your earliest convenience.\n\nThank you for your urgent assistance.\n\n"
    "Best regards,\nMaterial Control Department"
)

AIRCRAFT_TYPES = sorted({r["aircraft_type"] for r in DEFAULT_DB["aircraft_registry"]})
REGISTRATION_CATALOG = [r["registration"] for r in DEFAULT_DB["aircraft_registry"]]
AIRPORTS = list(AIRPORT_COORDS.keys())
PART_NUMBER_CATALOG = sorted(
    {it["part_number"] for k in DEFAULT_DB["fak_kits"] for it in k["contents"]}
    | {it["part_number"] for w in DEFAULT_DB["station_warehouses"] for it in w["contents"]}
    | {it["part_number"] for w in DEFAULT_DB["alliance_warehouses"] for it in w["contents"]}
    | {r["part_number"] for r in DEFAULT_DB["pooling_stock"]})


# ----------------------------------------------------------------------------
# 2. 영속화
# ----------------------------------------------------------------------------
def _migrate_db(db):
    for r in db.get("pooling_partners", []):
        r.setdefault("coverage_airports", "")
    for r in db.get("aircraft_registry", []):
        r.setdefault("operator", "대한항공")
    return db


def load_db():
    if os.path.exists(DB_PATH):
        try:
            with open(DB_PATH, "r", encoding="utf-8") as f:
                db = json.load(f)
            if db.get("_schema_version") == SCHEMA_VERSION:
                return _migrate_db(db)
            os.replace(DB_PATH, DB_PATH + ".bak")
        except Exception:
            pass
    db = copy.deepcopy(DEFAULT_DB)
    save_db(db)
    return db


def save_db(db):
    db["_schema_version"] = SCHEMA_VERSION
    with open(DB_PATH, "w", encoding="utf-8") as f:
        json.dump(db, f, ensure_ascii=False, indent=2)


if "db" not in st.session_state:
    st.session_state.db = load_db()
if "case" not in st.session_state:
    st.session_state.case = None
if "chat_log" not in st.session_state:
    st.session_state.chat_log = []


# ----------------------------------------------------------------------------
# 3. 조회 헬퍼 + 이송 Lead Time 계산
# ----------------------------------------------------------------------------
def norm(s):
    return (s or "").strip().upper()


def tel_href(contact):
    return re.sub(r"[^0-9+]", "", contact or "")


def find_one(records, **filters):
    for r in records:
        if all(norm(r.get(k)) == norm(v) for k, v in filters.items()):
            return r
    return None


def find_all(records, **filters):
    return [r for r in records if all(norm(r.get(k)) == norm(v) for k, v in filters.items())]


def _content_find(bundle, **filters):
    if not bundle:
        return None
    for it in bundle.get("contents", []):
        if all(norm(it.get(k)) == norm(v) for k, v in filters.items()):
            return it
    return None


def resolve_aircraft_info(db, registration):
    row = find_one(db["aircraft_registry"], registration=registration)
    if row:
        return row["aircraft_type"], row.get("operator", "대한항공")
    for r in db["aircraft_registry"]:
        if norm(r["aircraft_type"]) == norm(registration):
            return r["aircraft_type"], r.get("operator", "대한항공")
    return None, None


def airlines_operating_type(db, aircraft_type):
    return {norm(r["airline"]): r for r in db["fleet_operators"] if norm(r["aircraft_type"]) == norm(aircraft_type)}


def _route_duration(origin, dest):
    for o, d, base, freq, dur in _ROUTES:
        if {norm(o), norm(d)} == {norm(origin), norm(dest)}:
            return dur
    return None


def calculate_lead_time(origin, dest, operator="대한항공"):
    """이송 총 소요시간(h) = 운송(직항 or ICN 경유) + 조업(픽업/인계) + 카고 부킹 리드타임."""
    if norm(origin) == norm(dest):
        return round(2.0 + CARGO_BOOKING_HOURS.get(operator, 3.0) * 0.3, 1)  # 현지 제휴 pull
    direct = _route_duration(origin, dest)
    if direct:
        transit = direct
    else:
        d1 = _route_duration(origin, "ICN") or 8.0
        d2 = _route_duration("ICN", dest) or 8.0
        transit = d1 + d2 + 2.0  # ICN 허브 환적
    handling = 3.0
    booking = CARGO_BOOKING_HOURS.get(operator, 3.0)
    return round(transit + handling + booking, 1)


# ----------------------------------------------------------------------------
# 4. 7단계 조회 로직 — 전부 데이터 기반
# ----------------------------------------------------------------------------
def evaluate_step(step, db, registration, aircraft_type, part_number, airport, operator):
    if step == 1:
        kit = find_one(db["fak_kits"], aircraft_type=aircraft_type)
        item = _content_find(kit, part_number=part_number)
        if item:
            detail = {**item, "kit_name": kit.get("kit_name", "")}
            return True, (f"✅ **FAK 키트 보유** — {registration}({aircraft_type}) 탑재 {kit.get('kit_name', '표준 키트')}에 포함.\n"
                           f"- 자재: {item.get('part_name', part_number)} · 키트 내 수량: {item['qty']}개 · 현장: {airport}\n\n"
                           f"동일 기종은 모두 같은 키트를 기체와 함께 탑재 → {airport} 현장에서 즉시 사용 가능."), detail
        return False, f"❌ 이 자재는 {aircraft_type} 표준 FAK 키트(소모품) 구성품이 아닙니다.", None

    if step == 2:
        wh = find_one(db["station_warehouses"], airport=airport)
        item = _content_find(wh, aircraft_type=aircraft_type, part_number=part_number)
        if item:
            detail = {**item, "warehouse_name": wh.get("warehouse_name", ""), "airport": airport}
            return True, (f"✅ **로컬 Allocation 재고 확인** — {airport} {wh.get('warehouse_name', '창고')}\n"
                           f"- 자재: {item.get('part_name', part_number)} · 수량: {item['qty']}개\n\n"
                           f"{airport} 스테이션 창고 재고로 즉시 조치 가능."), detail
        return False, f"❌ {airport} 스테이션 자사 창고에는 해당 재고가 없습니다.", None

    if step == 3:
        # 타 공항 자사 창고 + 제휴 창고(Non-Pool) 전부 탐색 → Lead Time 최단 정렬.
        options = []
        for wh in db["station_warehouses"]:
            if norm(wh["airport"]) == norm(airport):
                continue
            it = _content_find(wh, aircraft_type=aircraft_type, part_number=part_number)
            if it:
                options.append({"kind": "자사 타 스테이션", "from": wh["airport"], "holder": wh.get("warehouse_name", "자사 창고"),
                                "qty": it["qty"], "lead": calculate_lead_time(wh["airport"], airport, operator)})
        for aw in db.get("alliance_warehouses", []):
            it = _content_find(aw, aircraft_type=aircraft_type, part_number=part_number)
            if it:
                options.append({"kind": "제휴 창고(Non-Pool)", "from": aw["airport"], "holder": aw.get("partner", "제휴처"),
                                "qty": it["qty"], "lead": calculate_lead_time(aw["airport"], airport, operator)})
        if options:
            options.sort(key=lambda x: x["lead"])
            best = options[0]
            detail = {"options": options, "best": best}
            return True, (f"✅ **이송 최적화 가능** — 최단 경로 추천: **{best['holder']}({best['from']})** · {best['qty']}개\n"
                           f"- 예상 이송 소요(Lead Time): 약 **{best['lead']}시간** · 유형: {best['kind']}\n\n"
                           f"총 {len(options)}개 경로 중 가장 빠른 이송안입니다(전 경로 비교는 아래 표)."), detail
        return False, "❌ 타 공항 자사 창고·제휴 창고(Non-Pool)에도 땡겨올 재고가 없습니다.", None

    if step == 4:
        candidates = find_all(db["pooling_stock"], aircraft_type=aircraft_type, part_number=part_number)
        for hit in candidates:
            pinfo = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            coverage = [c.strip() for c in pinfo.get("coverage_airports", "").split(",") if c.strip()]
            if not coverage or norm(airport) in [norm(c) for c in coverage]:
                detail = {**hit, "contact": pinfo.get("contact", ""), "email": pinfo.get("email", ""),
                          "coverage_airports": pinfo.get("coverage_airports", "")}
                return True, (f"✅ **Pooling 지원 가능 — {hit['partner']}**\n"
                               f"- 자재: {hit.get('part_name', part_number)} · 보유: {hit['location']} · 수량: {hit['qty']}개\n"
                               f"- 커버리지: {pinfo.get('coverage_airports', '전 공항')} · 연락처: {detail['contact']}\n\n"
                               f"{airport}은(는) 이 파트너의 서비스 지역이라 대여 요청이 가능합니다."), detail
        if candidates:
            hit = candidates[0]
            pinfo = find_one(db["pooling_partners"], partner=hit["partner"]) or {}
            return False, (f"❌ {airport}은(는) 이 자재 보유 Pooling 파트너의 서비스 지역이 아닙니다. "
                            f"(참고: {hit['partner']}가 {hit['location']}에 보유하나 커버리지 "
                            f"{pinfo.get('coverage_airports', '미등록')}에 {airport} 없음.)"), None
        return False, "❌ 해당 자재를 보유한 Pooling 파트너사가 없습니다.", None

    if step == 5:
        operators = airlines_operating_type(db, aircraft_type)
        station_airlines = find_all(db["main_station_airlines"], airport=airport)
        queried = [a for a in station_airlines if norm(a["airline"]) in operators]
        detail = {"queried_airlines": queried}
        if queried:
            names = ", ".join(a["airline"] for a in queried)
            return False, (f"ℹ️ {airport} 기반 항공사 중 {aircraft_type} 운영사({names})에 대여 문의가 필요합니다. "
                            f"타사 실재고는 사전에 알 수 없어 직접 문의로 확인합니다."), detail
        station_names = ", ".join(a["airline"] for a in station_airlines) or "등록된 항공사 없음"
        return False, (f"❌ {airport} Main Station 항공사({station_names}) 중 {aircraft_type} 운영사가 없습니다. 6단계로 넘어갑니다."), detail

    if step == 6:
        operators = list(airlines_operating_type(db, aircraft_type).values())
        detail = {"queried_airlines": operators}
        names = ", ".join(o["airline"] for o in operators) or "등록된 운영사 없음"
        return False, (f"ℹ️ {aircraft_type} 운영 항공사({names}) 전체에 대여 문의가 필요합니다. 타사 실재고는 기밀이라 개별 문의로 확인합니다."), detail

    if step == 7:
        flights = fetch_flight_schedule(airport)[:3]
        st_info = find_one(db["station_info"], airport=airport) or {}
        is_jinair = (norm(operator) == norm("진에어"))
        jinair_auth = (st_info.get("jinair_cargo_auth") == "Y")
        detail = {"flights": flights, "customs": db["customs_team"], "is_jinair": is_jinair,
                  "jinair_auth": jinair_auth, "st_info": st_info}
        return True, ("🧳 **본사 파송 (Hand-carry / Cargo)** — 최종 수단\n"
                       "가장 빠른 편명 예약 + 통관(야간 가이드) + AI 인보이스로 파송을 준비하세요."
                       + ("\n\n⚠️ **운영사=진에어**: 목적지 화물 인가 여부를 반드시 확인하세요." if is_jinair else "")), detail

    raise ValueError(f"알 수 없는 단계: {step}")


# ----------------------------------------------------------------------------
# 5. 실시간 항공편 / 공항 데이터 공급자 (더미)
# ----------------------------------------------------------------------------
FLIGHT_API_CONFIG = {"enabled": False, "base_url": "https://api.example-airline.com/v1/flights", "api_key": ""}
PARKING_API_CONFIG = {"enabled": False, "base_url": "https://api.example-airline.com/v1/stands", "api_key": ""}

_ROUTES = [
    ("ICN", "LAX", 11, 3, 11.2), ("ICN", "JFK", 81, 2, 14.0), ("ICN", "CDG", 901, 2, 12.5),
    ("ICN", "FRA", 905, 2, 11.7), ("ICN", "SIN", 643, 2, 6.5), ("ICN", "HKG", 603, 3, 3.7),
    ("ICN", "NRT", 705, 3, 2.5), ("ICN", "BKK", 651, 2, 5.9), ("ICN", "SYD", 121, 2, 10.5),
    ("ICN", "HND", 719, 2, 2.5), ("GMP", "HND", 2711, 2, 2.6),
]
_OFFSET_CYCLE = [-2.0, 6.0, -0.5, 8.0, 3.0, -4.0, 10.0, 1.5, -1.0, 5.0, 12.0, -3.0,
                 4.0, 9.0, -5.0, 7.0, 2.0, 11.0, -2.5, 6.5, 0.5, -6.0, 13.0, 3.5]
_PARKING_STAND_CAPACITY = {"ICN": 46, "GMP": 20, "NRT": 30, "HND": 28, "CDG": 60, "JFK": 55,
                           "SIN": 35, "HKG": 40, "FRA": 58, "LAX": 50, "BKK": 32, "SYD": 26}


def _generate_dummy_live_flights():
    now = datetime.now()
    flights, idx = [], 0
    for origin, dest, base, freq, dur in _ROUTES:
        for k in range(freq):
            for (o, d, no) in [(origin, dest, base + 2 * k), (dest, origin, base + 2 * k + 1)]:
                off = _OFFSET_CYCLE[idx % len(_OFFSET_CYCLE)]
                idx += 1
                dep = now + timedelta(hours=off)
                arr = dep + timedelta(hours=dur)
                if now < dep:
                    status, prog = "예정", 0.0
                elif now >= arr:
                    status, prog = "도착", 1.0
                else:
                    status = "비행 중"; prog = (now - dep).total_seconds() / (arr - dep).total_seconds()
                flights.append({"flight_no": f"KE{no:04d}", "airline": "대한항공", "origin": o, "destination_airport": d,
                                "dep_time": dep, "arr_time": arr, "status": status, "progress": round(prog, 3),
                                "hours_from_now": round(max((arr - now).total_seconds() / 3600, 0.0), 1)})
    return flights


@st.cache_data(ttl=60, show_spinner=False)
def _fetch_raw_flight_feed():
    if FLIGHT_API_CONFIG["enabled"]:
        raise NotImplementedError("FLIGHT_API_CONFIG.enabled=True면 실제 연동 로직을 구현하세요.")
    return _generate_dummy_live_flights()


def fetch_live_flights():
    return _fetch_raw_flight_feed()


def fetch_flight_schedule(destination_airport=None):
    flights = [f for f in fetch_live_flights() if f["status"] == "예정"]
    if destination_airport:
        flights = [f for f in flights if norm(f["destination_airport"]) == norm(destination_airport)]
    return sorted(flights, key=lambda f: f["hours_from_now"])


@st.cache_data(ttl=60, show_spinner=False)
def fetch_airport_parking_status():
    if PARKING_API_CONFIG["enabled"]:
        raise NotImplementedError("PARKING_API_CONFIG.enabled=True면 실제 연동 로직을 구현하세요.")
    import random
    rng = random.Random(datetime.now().strftime("%Y%m%d%H%M"))
    return {ap: {"total_stands": total, "occupied": rng.randint(int(total * 0.4), int(total * 0.95))}
            for ap, total in _PARKING_STAND_CAPACITY.items()}


# ----------------------------------------------------------------------------
# 6. AI 브리핑(즉시·규칙 기반) + 로컬 LLM(선택)
# ----------------------------------------------------------------------------
def build_rule_based_briefing(case):
    lines = [
        "### 📌 케이스 개요",
        f"- 기번: **{case['registration']}** → 기종: **{case['aircraft_type']}** · 운영사: **{case['operator']}**",
        f"- 자재: **{case['part_number']}** · 발생 공항: **{case['airport']}**",
        f"- 현재 단계: **{case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}**",
        "", "### 🔍 지금까지 조회 결과",
    ]
    for rec in case["records"]:
        lines.append(f"- {'✅' if rec['found'] else '❌'} {STEP_NAMES[rec['step']]}")
    lines.append("")
    lines.append("### 💡 추천 행동")
    last = case["records"][-1]
    step = case["step"]
    if case["finished"]:
        if last["found"]:
            act = {1: "현장 정비사에게 FAK 키트 보유를 재확인·전달하세요.",
                   2: f"{case['airport']} Allocation 부서에 창고 자재 반출을 요청하세요.",
                   3: "추천 이송 경로로 최단 이송 업체를 수배하세요.",
                   4: "Pooling 파트너사에 대여 절차를 진행하세요.",
                   7: "가장 빠른 편명으로 통관·파송을 확정하세요."}.get(last["step"], "확보 경로로 마무리하세요.")
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 확보 완료. {act}")
        else:
            lines.append(f"- 🏁 **{STEP_NAMES[last['step']]}**에서 담당자가 종료(외부 확보/대체).")
    elif last["found"]:
        lines.append(f"- 🎯 **{STEP_NAMES[step]}**에서 확보 가능. 아래에서 승인하면 진행합니다.")
    elif step in (5, 6):
        n = len(last["detail"].get("queried_airlines", []))
        lines.append(f"- ✉️ 문의 대상 **{n}개 항공사**에 영문 긴급 대여 메일을 보내세요.")
        lines.append("- 🛡️ 7단계 통관은 시간이 걸리므로 통관팀 사전 준비를 병행 권장.")
    elif step >= TOTAL_STEPS:
        lines.append("- ▶ 최종 파송 단계입니다. 통관·편명을 확정하세요.")
    else:
        lines.append(f"- ▶ 승인/거절에 따라 **{STEP_NAMES[step + 1]}**로 진행합니다.")
    return "\n".join(lines)


@st.cache_resource(show_spinner=False)
def _load_local_llm():
    from transformers import pipeline
    return pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")


def generate_llm_email_body(case):
    try:
        pipe = _load_local_llm()
        messages = [
            {"role": "system", "content": ("You are a material control officer at an airline. Write only the body of a "
                                            "polite, concise English business email requesting AOG spare-part support from "
                                            "another airline. 3-4 sentences, body only (no subject).")},
            {"role": "user", "content": (f"Aircraft type {case['aircraft_type']}, part {case['part_number']}, station "
                                         f"{case['airport']}. We are short of this part; ask for loan availability.")},
        ]
        out = pipe(messages, max_new_tokens=120, do_sample=False)
        return (out[0]["generated_text"][-1]["content"] or "").strip() or None
    except Exception:
        return None


# ----------------------------------------------------------------------------
# 7. 케이스 상태 관리
# ----------------------------------------------------------------------------
def try_start_case(registration, part_number, airport, mechanic_contact):
    aircraft_type, operator = resolve_aircraft_info(st.session_state.db, registration)
    if aircraft_type is None:
        return False, (f"기번 **{registration}**은(는) 등록부에 없습니다. '⚙️ 데이터 관리 → 🛩️ 기번 등록부'에 추가 후 다시 시도하세요.")
    case = {"registration": registration.strip(), "aircraft_type": aircraft_type, "operator": operator,
            "part_number": part_number.strip(), "airport": airport.strip().upper(),
            "mechanic_contact": mechanic_contact.strip(),
            "step": 1, "finished": False, "records": [], "action_log": [],
            "started_at": datetime.now().strftime("%Y-%m-%d %H:%M")}
    found, message, detail = evaluate_step(1, st.session_state.db, case["registration"], aircraft_type, case["part_number"], case["airport"], operator)
    case["records"].append({"step": 1, "found": found, "message": message, "detail": detail})
    st.session_state.case = case
    _auto_sweep_silent_steps()
    return True, None


def proceed_step():
    case = st.session_state.case
    if case["finished"] or case["step"] >= TOTAL_STEPS:
        return
    nxt = case["step"] + 1
    found, message, detail = evaluate_step(nxt, st.session_state.db, case["registration"], case["aircraft_type"], case["part_number"], case["airport"], case["operator"])
    case["step"] = nxt
    case["records"].append({"step": nxt, "found": found, "message": message, "detail": detail})


def _auto_sweep_silent_steps():
    # 1~4단계(내부 재고/최적화/Pooling 조회, 판단 불필요)는 재고 없으면 자동 진행.
    # 5단계는 문의 대상 없으면 자동 통과. 확보 지점·문의 대상 있는 5·6·7에서만 멈춤.
    case = st.session_state.case
    while not case["finished"] and case["step"] < TOTAL_STEPS:
        rec = case["records"][-1]
        if rec["found"]:
            break
        step = case["step"]
        if step in (1, 2, 3, 4):
            proceed_step(); continue
        if step == 5 and not rec["detail"].get("queried_airlines"):
            proceed_step(); continue
        break


def resolve_here():
    if st.session_state.case and not st.session_state.case["finished"]:
        st.session_state.case["finished"] = True


def _build_agent_report(case):
    checked = len(case["records"])
    action_section = build_rule_based_briefing(case).split("### 💡 추천 행동", 1)[-1].strip()
    return f"🔎 {case['registration']}({case['aircraft_type']}/{case['operator']}) · {case['airport']} — {checked}단계까지 확인.\n\n{action_section}"


# ----------------------------------------------------------------------------
# 8. 자연어 파싱 — 기번 / 파트넘버 / 공항
# ----------------------------------------------------------------------------
def parse_aog_message(text):
    upper = text.upper()
    compact = re.sub(r"\s+", "", upper)
    registration = next((r for r in REGISTRATION_CATALOG if r in compact), None)
    if not registration:
        m = re.search(r"\b(?:HL|LJ)[0-9A-Z]{3,4}\b", upper)
        if m:
            registration = m.group(0)
    if not registration:
        registration = next((t for t in AIRCRAFT_TYPES if norm(t).replace(" ", "") in compact), None)
    airport = None
    for ap, (_, _, name) in AIRPORT_COORDS.items():
        if re.search(rf"(?<![A-Z0-9]){ap}(?![A-Z0-9])", upper) or name in text:
            airport = ap
            break
    part_number = next((pn for pn in PART_NUMBER_CATALOG if pn in compact), None)
    if not part_number:
        m = re.search(r"\b[A-Z0-9]{2,}(?:-[A-Z0-9]+){1,}\b", upper)
        if m:
            part_number = m.group(0)
    mechanic = None
    m = re.search(r"(01[0-9]-?\d{3,4}-?\d{4})", text)
    if m:
        mechanic = m.group(1)
    return {"registration": registration, "part_number": part_number, "airport": airport, "mechanic_contact": mechanic}


# ----------------------------------------------------------------------------
# 9. GUI 공통 (CSS · 스텝퍼 · 헤더 · 이력)
# ----------------------------------------------------------------------------
_AOG_CSS = """
<style>
:root { --aog-navy:#0b2e4f; --aog-blue:#2a78d6; --aog-line:#e1e6ee; --aog-muted:#6b7684; }
.stApp { background:#f4f6f9; }
.block-container { padding-top:2.0rem; }
h1,h2,h3,h4 { color:var(--aog-navy); }
.aog-hero { background:linear-gradient(120deg,#0b2e4f 0%,#164b7e 100%); color:#fff;
  padding:16px 22px; border-radius:16px; margin-bottom:14px; box-shadow:0 6px 18px rgba(11,46,79,.18); }
.aog-hero .t { font-size:1.3rem; font-weight:800; }
.aog-hero .s { margin-top:4px; opacity:.85; font-size:.8rem; line-height:1.5; }
.aog-card { background:#fff; border:1px solid var(--aog-line); border-radius:14px; padding:12px 16px;
  margin-bottom:8px; box-shadow:0 1px 3px rgba(11,46,79,.05); }
.aog-kv { display:flex; flex-wrap:wrap; gap:20px; }
.aog-kv .k { font-size:.66rem; color:var(--aog-muted); text-transform:uppercase; letter-spacing:.05em; }
.aog-kv .v { font-size:1.02rem; font-weight:800; color:var(--aog-navy); margin-top:1px; }
.aog-pill { display:inline-block; padding:3px 12px; border-radius:999px; font-size:.72rem; font-weight:800; }
.aog-pill.ok { background:#e2f5e2; color:#0a7a0a; } .aog-pill.no { background:#eef1f5; color:#7a8492; }
.aog-pill.ask { background:#fff3d6; color:#9a6b00; } .aog-pill.warn { background:#fde2e2; color:#b21b1b; }
.aog-stepper { display:flex; margin:2px 0 14px; }
.aog-stepper .st { flex:1; text-align:center; position:relative; }
.aog-stepper .st:not(:first-child)::before { content:""; position:absolute; left:-50%; top:15px; width:100%; height:3px; background:var(--aog-line); z-index:0; }
.aog-stepper .st.on:not(:first-child)::before { background:var(--aog-blue); }
.aog-stepper .dot { width:32px; height:32px; border-radius:50%; margin:0 auto; display:flex; align-items:center;
  justify-content:center; font-weight:800; font-size:.82rem; position:relative; z-index:1; border:3px solid var(--aog-line); background:#fff; color:#9aa5b1; }
.aog-stepper .st.cur .dot { border-color:var(--aog-blue); background:var(--aog-blue); color:#fff; box-shadow:0 0 0 4px rgba(42,120,214,.18); }
.aog-stepper .st.found .dot { border-color:#0ca30c; background:#0ca30c; color:#fff; }
.aog-stepper .lbl { font-size:.6rem; color:var(--aog-muted); margin-top:5px; line-height:1.15; }
.aog-stepper .st.cur .lbl { color:var(--aog-navy); font-weight:800; }
[data-testid="stMetric"] { background:#fff; border:1px solid var(--aog-line); border-radius:12px; padding:10px 14px; box-shadow:0 1px 3px rgba(11,46,79,.05); }
[data-testid="stMetricLabel"] p { color:var(--aog-muted); font-weight:700; }
</style>
"""
_STEP_SHORT = {1: "FAK", 2: "로컬창고", 3: "이송최적화", 4: "Pooling", 5: "MainStn", 6: "동일기종", 7: "파송"}


def render_stepper(case):
    recs = {r["step"]: r for r in case["records"]}
    cur = case["step"]
    cells = []
    for s in range(1, TOTAL_STEPS + 1):
        rec = recs.get(s)
        classes = ["st"] + (["on"] if s <= cur else [])
        if rec is not None and rec["found"]:
            classes.append("found"); icon = "✓"
        elif s == cur and not case["finished"]:
            classes.append("cur"); icon = str(s)
        elif rec is not None:
            icon = "✕"
        else:
            icon = str(s)
        cells.append(f'<div class="{" ".join(classes)}"><div class="dot">{icon}</div><div class="lbl">{s}.{_STEP_SHORT[s]}</div></div>')
    st.markdown(f'<div class="aog-stepper">{"".join(cells)}</div>', unsafe_allow_html=True)


def render_case_header(case):
    last = case["records"][-1]
    if case["finished"] and last["found"]:
        pill = '<span class="aog-pill ok">✅ 자재 확보</span>'
    elif case["finished"]:
        pill = '<span class="aog-pill no">⏹ 종료</span>'
    elif last["found"]:
        pill = '<span class="aog-pill ok">✅ 확보 가능 · 승인 대기</span>'
    elif case["step"] in (5, 6):
        pill = '<span class="aog-pill ask">✉️ 타사 문의 필요</span>'
    elif case["step"] == 7:
        pill = '<span class="aog-pill warn">🧳 파송 단계</span>'
    else:
        pill = '<span class="aog-pill ask">🔎 조회 진행 중</span>'
    op_pill = ('<span class="aog-pill warn">진에어</span>' if norm(case["operator"]) == norm("진에어")
               else '<span class="aog-pill ok">대한항공</span>')
    st.markdown(
        f'<div class="aog-card"><div class="aog-kv">'
        f'<div><div class="k">기번 → 기종</div><div class="v">{case["registration"]} · {case["aircraft_type"]}</div></div>'
        f'<div><div class="k">운영사</div><div class="v">{op_pill}</div></div>'
        f'<div><div class="k">자재 파트넘버</div><div class="v">{case["part_number"]}</div></div>'
        f'<div><div class="k">발생 공항</div><div class="v">{case["airport"]}</div></div>'
        f'<div><div class="k">현재 상태</div><div class="v">{pill}</div></div>'
        f'</div></div>', unsafe_allow_html=True)
    render_stepper(case)


def render_history_panel(case):
    db = st.session_state.db
    pn = case["part_number"]
    sh = sorted([r for r in db.get("sourcing_history", []) if norm(r.get("part_number")) == norm(pn)], key=lambda r: r.get("date", ""), reverse=True)
    dh = sorted([r for r in db.get("defect_history", []) if norm(r.get("part_number")) == norm(pn)], key=lambda r: r.get("date", ""), reverse=True)
    st.markdown(f"### 📖 이 자재 참고자료 — `{pn}` · 과거에 어떻게 처리했는지")
    c1, c2 = st.columns(2)
    with c1:
        st.markdown("**🔁 과거 수배(빌린) 이력** — 어디서·어떤 방식으로 확보 · 성공/실패")
        if sh:
            ok = sum(1 for r in sh if str(r.get("result", "")).startswith("성공"))
            top = Counter(r.get("source", "") for r in sh).most_common(1)[0][0]
            st.caption(f"📌 과거 {len(sh)}건 · 성공 {ok}건 · 최다 수배처: **{top}**")
            df = pd.DataFrame(sh)[["date", "airport", "resolved_step", "source", "method", "result", "lead_time_hours"]]
            df.columns = ["일자", "공항", "확보 단계", "수배처", "방식", "결과", "소요(h)"]
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            st.caption("이 자재의 과거 수배 이력이 없습니다.")
    with c2:
        st.markdown("**🛠️ 과거 결함 조치 이력** — 교환/디퍼/리셋 · 단일 vs 패키지 · 전용 공구")
        if dh:
            top_res = ", ".join(f"{k}({v})" for k, v in Counter(r.get("resolution", "") for r in dh).most_common())
            tools = [r.get("tools_required", "") for r in dh if r.get("tools_required") and r.get("tools_required") != "표준 공구"]
            cap = f"📌 과거 {len(dh)}건 · 조치 분포: {top_res}"
            if tools:
                cap += f" · 전용공구 예: **{tools[0]}**"
            st.caption(cap)
            df = pd.DataFrame(dh)[["date", "registration", "defect", "resolution", "parts_scope", "tools_required", "result", "downtime_hours"]]
            df.columns = ["일자", "기번", "결함", "조치유형", "수배범위", "전용공구", "결과", "다운타임(h)"]
            st.dataframe(df, use_container_width=True, hide_index=True)
        else:
            st.caption("이 자재의 과거 결함 조치 이력이 없습니다.")


# ----------------------------------------------------------------------------
# 10. 대시보드 화면
# ----------------------------------------------------------------------------
def render_case_intake():
    if "chat_prefill" in st.session_state:
        prefill = st.session_state.pop("chat_prefill")
        for field, key in {"registration": "intake_reg", "part_number": "intake_pn",
                           "airport": "intake_airport", "mechanic_contact": "intake_mechanic"}.items():
            if prefill.get(field):
                st.session_state[key] = prefill[field]
    with st.expander("⌨️ 직접 입력 (채팅으로 일부만 인식됐거나 수동 입력 시)", expanded=not st.session_state.case):
        with st.form("case_form", clear_on_submit=False):
            c1, c2, c3 = st.columns(3)
            registration = c1.text_input("기번 (Registration)", placeholder="예: HL7702, LJ2201", key="intake_reg")
            part_number = c2.text_input("자재 파트넘버 (Part Number)", placeholder="예: FUEL-NOZ-A330-07", key="intake_pn")
            airport = c3.text_input("AOG 발생 공항 (Station)", placeholder="예: ICN", key="intake_airport")
            mechanic = st.text_input("현장 정비사 연락처 (선택)", placeholder="예: 010-1234-5678", key="intake_mechanic")
            submitted = st.form_submit_button("🚨 AOG 프로세스 시작", type="primary", use_container_width=True)
        if submitted:
            if not registration.strip() or not part_number.strip() or not airport.strip():
                st.warning("기번 / 자재 파트넘버 / 공항을 모두 입력해주세요.")
            else:
                ok, err = try_start_case(registration, part_number, airport, mechanic)
                if not ok:
                    st.error(err)
                    st.session_state.chat_log.append({"role": "assistant", "content": err})
                else:
                    c = st.session_state.case
                    st.session_state.chat_log.append({"role": "assistant",
                        "content": f"시작합니다 — 기번 **{c['registration']}**({c['aircraft_type']}/{c['operator']}) · 자재 **{c['part_number']}** · 공항 **{c['airport']}**."})
                    st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(c)})
                    st.rerun()


def render_ai_panel():
    st.markdown("## 🤖 AI 상황 요약 & 추천 행동")
    case = st.session_state.case
    if not case:
        st.info("채팅 또는 직접 입력으로 케이스를 시작하면 이곳에 추천 행동이 즉시 표시됩니다.")
        return
    with st.container(border=True):
        st.markdown(build_rule_based_briefing(case))


def render_process_panel():
    st.markdown("## 📋 프로세스 진행")
    case = st.session_state.case
    if not case:
        st.info("케이스가 시작되지 않았습니다.")
        return
    last = case["records"][-1]
    with st.container(border=True):
        st.markdown(f"#### 진행 단계 {case['step']}/{TOTAL_STEPS} — {STEP_NAMES[case['step']]}")
        (st.success if last["found"] else st.warning)(last["message"])
        is_last = case["step"] >= TOTAL_STEPS
        bc1, bc2 = st.columns(2)
        if not case["finished"] and not is_last:
            if bc1.button("➡️ 아니요, 다음 대안 찾기", use_container_width=True, key="proceed_btn"):
                proceed_step(); _auto_sweep_silent_steps()
                st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(st.session_state.case)})
                st.rerun()
        if not case["finished"]:
            if case["step"] in (5, 6):
                label = "✅ 대여 확보됨 (프로세스 종료)"
            elif is_last:
                label = "✅ 파송 조치 확정 (프로세스 종료)"
            else:
                label = "✅ 네, 진행해주세요"
            if bc2.button(label, use_container_width=True, key="resolve_btn", type="primary"):
                resolve_here(); st.rerun()
    render_action_panel(case)
    with st.expander("📜 처리 이력 (Audit Log)", expanded=False):
        for rec in case["records"]:
            st.markdown(f"**{'✅' if rec['found'] else '❌'} {STEP_NAMES[rec['step']]}**\n\n{rec['message']}")
            st.divider()
        for entry in case["action_log"]:
            st.markdown(f"🗂️ {entry}")


def render_bulk_email_action(case, airlines, key_prefix):
    recipients = [a["email"] for a in airlines if a.get("email")]
    subject = DEFAULT_EMAIL_SUBJECT.format(aircraft_type=case["aircraft_type"], part_number=case["part_number"], airport=case["airport"])
    body_key = f"emailbody_{key_prefix}_{case['part_number']}_{case['aircraft_type']}_{case['airport']}"
    if body_key not in st.session_state:
        st.session_state[body_key] = DEFAULT_EMAIL_BODY_TEMPLATE.format(airport=case["airport"], aircraft_type=case["aircraft_type"], part_number=case["part_number"])
    st.write(f"문의 대상({len(airlines)}곳): {', '.join(a['airline'] for a in airlines) or '없음'}")
    if recipients:
        mailto = "mailto:" + ",".join(recipients) + "?subject=" + urllib.parse.quote(subject) + "&body=" + urllib.parse.quote(st.session_state[body_key])
        st.markdown(f'<a href="{mailto}" target="_blank">✉️ 문의 대상 전체({len(recipients)}곳)에 지금 바로 메일 작성 (영문)</a>', unsafe_allow_html=True)
    else:
        st.warning("등록된 이메일이 없습니다. '데이터 관리'에서 항공사 이메일을 추가하세요.")
    with st.expander("메일 내용 확인/수정 (영문 템플릿으로 바로 발송 가능)"):
        st.text_area("Body", key=body_key, height=160, label_visibility="collapsed")
        if HAS_TRANSFORMERS:
            if st.button("✨ AI로 다듬기 (영문·선택)", key=f"ai_polish_{key_prefix}"):
                with st.spinner("AI가 영문 문구를 다듬는 중..."):
                    polished = generate_llm_email_body(case)
                if polished:
                    st.session_state[body_key] = polished; st.rerun()
                else:
                    st.warning("AI 응답을 못 받았습니다. 기본 영문 문구로 발송하세요.")
        else:
            st.caption("ℹ️ transformers 미설치 — 기본 영문 템플릿으로 바로 발송 가능합니다.")
    if st.button("✉️ 긴급 메일 발송 완료로 기록", key=f"log_email_{key_prefix}"):
        case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {len(recipients)}개 항공사에 긴급 메일(영문) 발송 완료")
        st.rerun()


def _build_invoice_text(case, st_info, flight_no):
    today = datetime.now().strftime("%Y-%m-%d")
    return (
        "AOG SPARE PART SHIPPING INVOICE / DECLARATION\n"
        "--------------------------------------------------------\n"
        f"AOG Ref      : AOG-{today}-{case['registration']}\n"
        f"Aircraft     : {case['registration']} ({case['aircraft_type']})  |  Operator: {case['operator']}\n"
        f"Part Number  : {case['part_number']}    Qty: 1\n"
        "Reason       : Aircraft On Ground (AOG) - urgent spare, nil commercial value\n"
        f"Consignee    : {st_info.get('handling_agent', 'TBD')}\n"
        f"Address      : {st_info.get('address', 'TBD - verify station address')}\n"
        f"Destination  : {case['airport']}\n"
        f"Flight       : {flight_no or 'TBD (book on KE Cargo)'}\n"
        "Shipper      : Korean Air Material Control, ICN\n"
        "Customs      : AOG duty exemption requested. DECLARATION REQUIRED before handover.\n"
        "--------------------------------------------------------\n"
        "** 신고 누락 시 파송 지연 — 반드시 통관 신고 완료 후 조업사 인계 **"
    )


def render_customs_and_shipping(case, detail):
    st_info = detail.get("st_info", {})
    now_hour = datetime.now().hour
    is_night = now_hour >= 18 or now_hour < 9

    # 진에어 화물 인가 경고
    if detail.get("is_jinair"):
        if detail.get("jinair_auth"):
            st.success(f"✅ **[진에어 화물 인가 유효]** {case['airport']}은 진에어 Cargo Auth가 있어 자사 화물 파송이 가능합니다.")
        else:
            st.error(f"⚠️ **[진에어 화물 인가 불가]** {case['airport']}은 진에어 Cargo Auth가 없어 자사 화물기 파송 불가! "
                     f"→ **Hand-carry** 또는 **타사/우회 이송 업체** 수배가 필수입니다.")

    # 야간 통관 가이드
    with st.expander(f"🌙 야간 통관 가이드 (관세사 부재 시 직접 수행) {'· 현재 야간시간대' if is_night else ''}", expanded=is_night):
        st.caption("야간에는 통관팀이 퇴근해 담당자가 직접 통관을 진행해야 합니다. 아래 체크리스트로 휴먼에러(신고 누락)를 방지하세요.")
        for line in st.session_state.db.get("night_customs_manual", []):
            st.markdown(f"- {line}")

    # AI 인보이스(수취인 자동 채움 → 오배송 방지)
    with st.expander("🧾 AI 파송 인보이스 생성 (조업사 주소 자동 반영 · 오배송 방지)", expanded=False):
        st.caption(f"수취인(조업사): **{st_info.get('handling_agent', '미등록')}** / 주소: {st_info.get('address', '미등록 — 데이터 관리에서 갱신')}")
        st.caption("조업사가 바뀌면 주소가 바뀌어 오배송(휴먼에러)이 납니다. DB에 등록된 검증 주소를 그대로 인보이스에 바인딩합니다.")
        flights = detail.get("flights", [])
        flight_no = flights[0]["flight_no"] if flights else None
        inv_key = f"invoice_{case['registration']}_{case['part_number']}_{case['airport']}"
        if st.button("✨ 인보이스 생성/갱신", key=f"gen_inv_{case['airport']}"):
            st.session_state[inv_key] = _build_invoice_text(case, st_info, flight_no)
        if inv_key not in st.session_state:
            st.session_state[inv_key] = _build_invoice_text(case, st_info, flight_no)
        st.text_area("Invoice / Declaration", key=inv_key, height=280, label_visibility="collapsed")


def render_action_panel(case):
    st.markdown("#### 🎯 다음 조치")
    last = case["records"][-1]
    step, found, detail = last["step"], last["found"], last["detail"]

    if step == 1 and found:
        st.write(f"🧰 {case['aircraft_type']} {detail.get('kit_name', 'FAK')} · {detail.get('part_name', case['part_number'])} · 현장 {case['airport']} · {detail['qty']}개")
        if case["mechanic_contact"]:
            st.markdown(f'<a href="tel:{tel_href(case["mechanic_contact"])}">📞 현장 정비사({case["mechanic_contact"]})에게 전화</a>', unsafe_allow_html=True)
        else:
            st.caption("정비사 연락처가 없습니다. 위 정보로 직접 전달하세요.")
        if st.button("☎️ 정비사 전달 완료로 기록", key="log_step1"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 현장 정비사에게 FAK 키트 자재 안내 완료"); st.rerun()

    elif step == 2 and found:
        dept = find_one(st.session_state.db["allocation_dept_contacts"], airport=case["airport"])
        st.write(f"📦 {detail.get('warehouse_name', '창고')} ({case['airport']}) · {detail.get('part_name', case['part_number'])} · {detail['qty']}개")
        if dept:
            st.write(f"담당: **{dept['department']}**")
            c1, c2 = st.columns(2)
            c1.markdown(f'<a href="tel:{tel_href(dept["contact"])}">📞 {dept["contact"]}</a>', unsafe_allow_html=True)
            c2.markdown(f'<a href="mailto:{dept["email"]}">✉️ {dept["email"]}</a>', unsafe_allow_html=True)
        else:
            st.warning(f"{case['airport']} Allocation 부서 연락처가 없습니다. '데이터 관리'에서 추가하세요.")
        if st.button("☎️ Allocation 부서 연락 완료로 기록", key="log_step2"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {case['airport']} Allocation 부서 연락 완료"); st.rerun()

    elif step == 3 and found:
        opts = detail["options"]; best = detail["best"]
        st.write(f"🚚 최적 이송 추천: **{best['holder']}({best['from']})** · {best['qty']}개 · 예상 **{best['lead']}시간**")
        df = pd.DataFrame(opts)[["kind", "from", "holder", "qty", "lead"]]
        df.columns = ["유형", "출발공항", "보유처", "수량", "예상소요(h)"]
        st.dataframe(df, use_container_width=True, hide_index=True,
                     column_config={"예상소요(h)": st.column_config.NumberColumn("예상소요(h)", format="%.1f")})
        st.caption("소요시간 = 운송(직항/ICN 경유) + 조업 + 카고 부킹 리드타임. 가장 짧은 경로가 최적안입니다.")
        if st.button("🚚 최적 경로로 이송 수배 완료로 기록", key="log_step3"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 이송 수배: {best['holder']}({best['from']}) → {case['airport']} (예상 {best['lead']}h)"); st.rerun()

    elif step == 4 and found:
        st.write(f"🤝 {detail.get('part_name', case['part_number'])} · {detail['location']} · {detail['qty']}개 · 커버리지 {detail.get('coverage_airports', '전 공항')}")
        c1, c2 = st.columns(2)
        c1.markdown(f'<a href="tel:{tel_href(detail["contact"])}">📞 {detail["contact"]}</a>', unsafe_allow_html=True)
        if detail.get("email"):
            c2.markdown(f'<a href="mailto:{detail["email"]}">✉️ {detail["email"]}</a>', unsafe_allow_html=True)
        if st.button("☎️ Pooling 파트너사 연락 완료로 기록", key="log_step4"):
            case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] {detail['partner']}에 대여 연락 완료"); st.rerun()

    elif step in (5, 6):
        if step == 5:
            st.info("발생 공항 기반 항공사 중 우리 기종 운영사에 우선 문의합니다(가까워 빠른 지원 기대).")
        else:
            st.info("기종을 운영하는 모든 타 항공사에 문의합니다. 타사 실재고는 기밀이라 개별 문의로만 확인됩니다.")
        render_bulk_email_action(case, detail.get("queried_airlines", []), f"step{step}")

    elif step == 7:
        customs = detail["customs"]
        st.write(f"통관팀: **{customs['name']}** · {customs['contact']} · {customs['email']}")
        flights = detail["flights"]
        if flights:
            opts = [f"{f['flight_no']} — 약 {f['hours_from_now']:.1f}시간 후 도착 [{(datetime.now() + timedelta(hours=f['hours_from_now'])).strftime('%m/%d %H:%M')}]" for f in flights]
            chosen = st.radio("Hand-carry/Cargo 편명 선택 (KE Cargo 편명 조회 반영, 가장 빠른 편 상단)", opts, key="flight_choice")
            if st.button("✅ 이 편으로 파송 확정", key="log_step7"):
                case["action_log"].append(f"[{datetime.now().strftime('%H:%M')}] 통관 연계·파송 편명 확정: {chosen}"); st.rerun()
        else:
            st.warning(f"{case['airport']}행 예정편이 조회되지 않습니다. '실시간 운항 현황'에서 경유편을 확인하세요.")
        st.divider()
        render_customs_and_shipping(case, detail)


def render_dashboard_page():
    st.markdown('<div class="aog-hero"><div class="t">🚨 AOG 자재 수급 어시스턴트</div>'
                '<div class="s">기번·자재 파트넘버·공항을 말하면 데이터를 근거로 7단계를 순차 조회합니다. '
                'FAK→로컬창고→이송최적화→Pooling은 자동으로 훑고, 타사 문의(5·6)와 파송(7)에서만 승인을 기다립니다.</div></div>',
                unsafe_allow_html=True)
    for msg in st.session_state.chat_log:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])
    render_case_intake()
    if st.session_state.case:
        render_case_header(st.session_state.case)
        left, right = st.columns(2)
        with left:
            render_ai_panel()
        with right:
            render_process_panel()
        st.divider()
        render_history_panel(st.session_state.case)
        st.divider()
        if st.button("🔄 새 AOG 건 시작 (초기화)"):
            st.session_state.case = None
            st.session_state.chat_log = []
            st.rerun()
    user_msg = st.chat_input("예: HL7702 CDG에서 AOG, 부품 FUEL-NOZ-A330-07 필요 / LJ2201 CDG에서 AOG, HYD-PUMP-737-11")
    if user_msg:
        st.session_state.chat_log.append({"role": "user", "content": user_msg})
        parsed = parse_aog_message(user_msg)
        label = {"registration": "기번", "part_number": "자재 파트넘버", "airport": "공항"}
        missing = [k for k in ("registration", "part_number", "airport") if not parsed[k]]
        if missing:
            recog = ", ".join(f"{label[k]} **{v}**" for k, v in parsed.items() if v and k in label)
            reply = ((f"인식: {recog}\n\n" if recog else "") + f"**{', '.join(label[k] for k in missing)}**를 못 찾았어요. 아래 '직접 입력'에서 마저 채워주세요.")
            st.session_state.chat_log.append({"role": "assistant", "content": reply})
            st.session_state.chat_prefill = parsed
        else:
            ok, err = try_start_case(parsed["registration"], parsed["part_number"], parsed["airport"], parsed.get("mechanic_contact") or "")
            if not ok:
                st.session_state.chat_log.append({"role": "assistant", "content": err})
                st.session_state.chat_prefill = parsed
            else:
                c = st.session_state.case
                st.session_state.chat_log.append({"role": "assistant",
                    "content": f"인식했습니다 — 기번 **{c['registration']}**({c['aircraft_type']}/{c['operator']}) · 자재 **{c['part_number']}** · 공항 **{c['airport']}**. FAK부터 순차 조회할게요."})
                st.session_state.chat_log.append({"role": "assistant", "content": _build_agent_report(c)})
        st.rerun()


# ----------------------------------------------------------------------------
# 11. 데이터 관리 화면
# ----------------------------------------------------------------------------
def df_editor_save(key, columns, int_cols=()):
    db = st.session_state.db
    df = pd.DataFrame(db.get(key, []))
    if df.empty:
        df = pd.DataFrame(columns=columns)
    edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"editor_{key}")
    if st.button("💾 저장", key=f"save_{key}"):
        cleaned = edited.fillna("")
        for c in int_cols:
            if c in cleaned.columns:
                cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
        db[key] = cleaned.to_dict("records")
        save_db(db); st.success("저장되었습니다.")


def render_bundle_editor(db_key, group_field, group_label, name_field, content_cols, int_cols=()):
    db = st.session_state.db
    bundles = db.setdefault(db_key, [])
    if bundles:
        def fmt(i):
            b = bundles[i]
            nm = f" · {b.get(name_field, '')}" if name_field else ""
            return f"{b.get(group_field, '?')}{nm} ({len(b.get('contents', []))}종)"
        idx = st.selectbox(f"{group_label} 선택", list(range(len(bundles))), format_func=fmt, key=f"sel_{db_key}")
        bundle = bundles[idx]
        if name_field:
            bundle[name_field] = st.text_input("묶음 이름", value=bundle.get(name_field, ""), key=f"name_{db_key}_{idx}")
        df = pd.DataFrame(bundle.get("contents", []))
        if df.empty:
            df = pd.DataFrame(columns=content_cols)
        edited = st.data_editor(df, num_rows="dynamic", use_container_width=True, key=f"ed_{db_key}_{idx}")
        if st.button("💾 이 묶음 저장", key=f"save_{db_key}_{idx}"):
            cleaned = edited.fillna("")
            for c in int_cols:
                if c in cleaned.columns:
                    cleaned[c] = pd.to_numeric(cleaned[c], errors="coerce").fillna(0).astype(int)
            bundle["contents"] = cleaned.to_dict("records")
            save_db(db); st.success("저장되었습니다.")
    else:
        st.info("등록된 묶음이 없습니다.")
    with st.expander(f"➕ 새 {group_label} 추가"):
        newg = st.text_input(f"{group_field} 값", key=f"newg_{db_key}")
        newn = st.text_input("묶음 이름", key=f"newn_{db_key}") if name_field else None
        if st.button("추가", key=f"add_{db_key}"):
            if newg.strip():
                nb = {group_field: newg.strip(), "contents": []}
                if name_field:
                    nb[name_field] = (newn or "").strip()
                bundles.append(nb); save_db(db); st.rerun()


def render_admin_page():
    st.markdown('<div class="aog-hero"><div class="t">⚙️ 데이터 관리</div>'
                '<div class="s">여기 저장된 데이터가 프로세스 판단의 유일한 근거입니다. 저장 즉시 대시보드 조회에 반영됩니다.</div></div>',
                unsafe_allow_html=True)
    if os.path.exists(DB_PATH):
        st.caption(f"마지막 저장: {datetime.fromtimestamp(os.path.getmtime(DB_PATH)).strftime('%Y-%m-%d %H:%M')}")
    tabs = st.tabs(["🛩️ 기번 등록부", "🧰 FAK 키트", "📦 로컬 Allocation 창고", "🌐 제휴 창고(Non-Pool)", "🤝 Pooling",
                    "🛫 Main Station", "✈️ 기종별 운영사", "🏢 공항 정보(조업사·진에어인가)", "🌙 야간 통관 가이드",
                    "📇 Allocation·통관팀", "🔁 수배 이력", "🛠️ 결함 조치 이력"])
    with tabs[0]:
        st.caption("기번 → 기종·운영사 매핑. 운영사(대한항공/진에어)에 따라 7단계 화물 파송 로직이 분기됩니다.")
        df_editor_save("aircraft_registry", ["registration", "aircraft_type", "operator"])
    with tabs[1]:
        st.caption("기종별 동일 키트 하나(소모품 패키지). 로터블(교환품)은 넣지 마세요 → 넣으면 1단계에서 오탐됩니다.")
        render_bundle_editor("fak_kits", "aircraft_type", "FAK 키트(기종)", "kit_name", ["part_number", "part_name", "qty"], int_cols=["qty"])
    with tabs[2]:
        st.caption("공항별 자사 창고. 발생 공항 창고에 있어야 2단계에서 즉시 불출됩니다. 타 공항 재고는 3단계 이송 대상.")
        render_bundle_editor("station_warehouses", "airport", "자사 창고(공항)", "warehouse_name", ["aircraft_type", "part_number", "part_name", "qty"], int_cols=["qty"])
    with tabs[3]:
        st.caption("풀 계약은 없지만 제휴로 땡겨올 수 있는 외부 창고(Non-Pool). 3단계 이송 최적화 탐색 대상입니다.")
        render_bundle_editor("alliance_warehouses", "airport", "제휴 창고(공항)", "partner", ["aircraft_type", "part_number", "part_name", "qty"], int_cols=["qty"])
    with tabs[4]:
        st.markdown("**파트너사 목록** (`coverage_airports`: 지원 가능 공항 콤마 구분)")
        df_editor_save("pooling_partners", ["partner", "contact", "email", "coverage_airports"])
        st.markdown("**파트너사 보유 재고**")
        df_editor_save("pooling_stock", ["partner", "aircraft_type", "part_number", "part_name", "qty", "location"], int_cols=["qty"])
    with tabs[5]:
        df_editor_save("main_station_airlines", ["airport", "airline", "contact", "email"])
    with tabs[6]:
        df_editor_save("fleet_operators", ["aircraft_type", "airline", "contact", "email"])
    with tabs[7]:
        st.caption("공항별 조업사·검증 주소(인보이스 자동화)·진에어 화물 인가(Y/N). 조업사 변경 시 주소를 갱신해 오배송을 방지합니다.")
        df_editor_save("station_info", ["airport", "jinair_cargo_auth", "handling_agent", "address"])
    with tabs[8]:
        st.caption("야간(관세사 부재) 통관 시 담당자가 직접 수행하는 체크리스트. 한 줄에 한 항목.")
        db = st.session_state.db
        txt = st.text_area("야간 통관 체크리스트", value="\n".join(db.get("night_customs_manual", [])), height=200, key="night_manual_edit")
        if st.button("💾 저장", key="save_night_manual"):
            db["night_customs_manual"] = [ln for ln in txt.splitlines() if ln.strip()]
            save_db(db); st.success("저장되었습니다.")
    with tabs[9]:
        st.markdown("**공항별 Allocation 부서 연락처**")
        df_editor_save("allocation_dept_contacts", ["airport", "department", "contact", "email"])
        st.markdown("**통관팀 연락처**")
        db = st.session_state.db
        c1, c2, c3 = st.columns(3)
        name = c1.text_input("담당팀명", value=db["customs_team"]["name"], key="customs_name")
        contact = c2.text_input("연락처", value=db["customs_team"]["contact"], key="customs_contact")
        email = c3.text_input("이메일", value=db["customs_team"]["email"], key="customs_email")
        if st.button("💾 저장", key="save_customs"):
            db["customs_team"] = {"name": name, "contact": contact, "email": email}
            save_db(db); st.success("저장되었습니다.")
    with tabs[10]:
        st.caption("자재별 과거 수배(빌린) 이력 — 어디서·어떤 방식으로 확보했고 성공/실패했는지.")
        df_editor_save("sourcing_history", ["date", "registration", "aircraft_type", "part_number", "airport", "resolved_step", "source", "method", "result", "lead_time_hours"], int_cols=["lead_time_hours"])
    with tabs[11]:
        st.caption("자재별 과거 결함 조치 이력 — 조치유형/수배범위/전용공구.")
        df_editor_save("defect_history", ["date", "registration", "aircraft_type", "part_number", "defect", "resolution", "parts_scope", "tools_required", "result", "downtime_hours"], int_cols=["downtime_hours"])


# ----------------------------------------------------------------------------
# 12. 실시간 운항 현황
# ----------------------------------------------------------------------------
def build_flight_map(flights, parking, movements):
    fig = go.Figure()
    route_freq = Counter()
    for f in flights:
        route_freq[tuple(sorted([f["origin"], f["destination_airport"]]))] += 1
    max_rf = max(route_freq.values()) if route_freq else 1
    for (a, b), c in route_freq.items():
        if a not in AIRPORT_COORDS or b not in AIRPORT_COORDS:
            continue
        la, lo, _ = AIRPORT_COORDS[a]; lb, lob, _ = AIRPORT_COORDS[b]
        fig.add_trace(go.Scattergeo(lat=[la, lb], lon=[lo, lob], mode="lines",
            line=dict(width=1 + 5 * c / max_rf, color="#2a78d6"), opacity=0.4, hoverinfo="skip", showlegend=False))
    max_mv = max(movements.values()) if movements else 1
    lats, lons, texts, sizes, colors = [], [], [], [], []
    for code, (lat, lon, name) in AIRPORT_COORDS.items():
        mv = movements.get(code, 0)
        info = parking.get(code, {"total_stands": 0, "occupied": 0})
        rate = info["occupied"] / info["total_stands"] if info["total_stands"] else 0
        lats.append(lat); lons.append(lon)
        texts.append(f"{name}({code})<br>운항량 {mv}편 · 정박 {info['occupied']}/{info['total_stands']} ({rate:.0%})")
        sizes.append(10 + 26 * (mv / max_mv)); colors.append(rate)
    fig.add_trace(go.Scattergeo(lat=lats, lon=lons, text=texts, mode="markers", hoverinfo="text", name="공항",
        marker=dict(size=sizes, color=colors, colorscale=[[0, "#0ca30c"], [0.7, "#fab219"], [1, "#d03b3b"]],
                    cmin=0, cmax=1, showscale=True, colorbar=dict(title="정박<br>혼잡도", tickformat=".0%"),
                    line=dict(width=1, color="#ffffff"))))
    for f in flights:
        if f["status"] != "비행 중" or f["origin"] not in AIRPORT_COORDS or f["destination_airport"] not in AIRPORT_COORDS:
            continue
        olat, olon, _ = AIRPORT_COORDS[f["origin"]]; dlat, dlon, _ = AIRPORT_COORDS[f["destination_airport"]]
        plat = olat + (dlat - olat) * f["progress"]; plon = olon + (dlon - olon) * f["progress"]
        fig.add_trace(go.Scattergeo(lat=[plat], lon=[plon], mode="markers", showlegend=False, hoverinfo="text",
            text=f"{f['flight_no']} {f['origin']}→{f['destination_airport']} ({f['progress']:.0%})",
            marker=dict(size=9, symbol="triangle-up", color="#eb6834", line=dict(width=1, color="#ffffff"))))
    fig.update_geos(projection_type="natural earth", showland=True, landcolor="#f0efec",
                    showcountries=True, countrycolor="#c3c2b7", showocean=True, oceancolor="#fcfcfb")
    fig.update_layout(height=560, margin=dict(l=0, r=0, t=10, b=0), font=dict(family="system-ui, -apple-system, 'Segoe UI', sans-serif"))
    return fig


_STATUS_ICON = {"비행 중": "🛫 비행 중", "예정": "🕐 예정", "도착": "✅ 도착"}
_STATUS_ORDER = {"비행 중": 0, "예정": 1, "도착": 2}


def render_map_page():
    st.markdown('<div class="aog-hero"><div class="t">🗺️ 실시간 운항 & 허브 현황</div>'
                '<div class="s">어느 공항이 허브이고 어떤 노선이 주력인지 한눈에. 더미 데이터이며 실 API 연동 시 코드 변경 없이 반영됩니다.</div></div>',
                unsafe_allow_html=True)
    if st.button("🔄 새로고침"):
        _fetch_raw_flight_feed.clear(); fetch_airport_parking_status.clear(); st.rerun()
    flights = fetch_live_flights()
    parking = fetch_airport_parking_status()
    movements = Counter(); route_freq = Counter()
    for f in flights:
        movements[f["origin"]] += 1; movements[f["destination_airport"]] += 1
        route_freq[tuple(sorted([f["origin"], f["destination_airport"]]))] += 1
    in_flight = sum(1 for f in flights if f["status"] == "비행 중")
    top_hub = movements.most_common(1)[0] if movements else ("-", 0)
    top_route = route_freq.most_common(1)[0] if route_freq else (("-", "-"), 0)
    hub_name = AIRPORT_COORDS.get(top_hub[0], (0, 0, top_hub[0]))[2]
    k1, k2, k3, k4 = st.columns(4)
    k1.metric("전체 항공편", f"{len(flights)}편")
    k2.metric("🛫 운항 중", f"{in_flight}편")
    k3.metric("최다 허브", f"{hub_name}({top_hub[0]})", f"{top_hub[1]}편 취항")
    k4.metric("주력 노선", f"{top_route[0][0]}–{top_route[0][1]}", f"{top_route[1]}편")
    st.plotly_chart(build_flight_map(flights, parking, movements), use_container_width=True)
    col1, col2 = st.columns(2)
    with col1:
        st.markdown("#### 🏆 공항별 운항량 (허브 순위)")
        rows = [{"공항": f"{AIRPORT_COORDS.get(a, (0, 0, a))[2]}({a})", "운항량(편)": c, "비중": c / sum(movements.values()) if movements else 0} for a, c in movements.most_common()]
        st.dataframe(pd.DataFrame(rows), use_container_width=True, hide_index=True,
                     column_config={"비중": st.column_config.ProgressColumn("비중", min_value=0, max_value=max([r["비중"] for r in rows] + [1]), format="%.0f%%")})
    with col2:
        st.markdown("#### 🔗 주력 노선 Top")
        rrows = [{"노선": f"{a}–{b}", "편수": c, "비중": c / sum(route_freq.values()) if route_freq else 0} for (a, b), c in route_freq.most_common(10)]
        st.dataframe(pd.DataFrame(rrows), use_container_width=True, hide_index=True,
                     column_config={"비중": st.column_config.ProgressColumn("비중", min_value=0, max_value=max([r["비중"] for r in rrows] + [1]), format="%.0f%%")})
    st.markdown("#### ✈️ 실시간 항공편")
    df = pd.DataFrame(flights)
    df["_o"] = df["status"].map(_STATUS_ORDER)
    df = df.sort_values(["_o", "hours_from_now"])
    df["상태"] = df["status"].map(_STATUS_ICON)
    df["출발"] = df["dep_time"].dt.strftime("%m/%d %H:%M")
    df["도착 예정"] = df["arr_time"].dt.strftime("%m/%d %H:%M")
    show = df[["flight_no", "origin", "destination_airport", "상태", "progress", "출발", "도착 예정"]]
    show.columns = ["편명", "출발", "도착", "상태", "진행률", "출발시각", "도착예정"]
    st.dataframe(show, use_container_width=True, hide_index=True,
                 column_config={"진행률": st.column_config.ProgressColumn("진행률", min_value=0, max_value=1, format="%.0f%%")})


# ----------------------------------------------------------------------------
# 13. 사이드바 내비게이션
# ----------------------------------------------------------------------------
st.markdown(_AOG_CSS, unsafe_allow_html=True)
with st.sidebar:
    st.markdown("## ✈️ AOG 어시스턴트")
    page = st.radio("화면 선택", ["🚨 AOG 대시보드", "🗺️ 실시간 운항 현황", "⚙️ 데이터 관리"], key="nav_page")
    st.divider()
    st.caption("⚡ 모든 판단은 '데이터 관리'의 데이터로만. 타사 문의 메일은 영문이 기본입니다.")
    st.caption("7단계: FAK→로컬창고→이송최적화→Pooling→MainStn→동일기종→파송")

if page == "🚨 AOG 대시보드":
    render_dashboard_page()
elif page == "🗺️ 실시간 운항 현황":
    render_map_page()
else:
    render_admin_page()


## Cell 3 — 앱 실행 (Colab 자체 포트포워딩만 사용, 외부 터널 불필요)

npm/localtunnel 같은 제3자 서비스를 전혀 쓰지 않고, Google Colab이 공식 제공하는 커널 포트포워딩(`google.colab.output`)만 사용합니다. 이미 접속 중인 `colab.research.google.com` 도메인을 그대로 경유하므로 사내망 보안 정책에 걸릴 일이 없습니다.

In [ ]:
import subprocess, time

LOG_PATH = "/content/streamlit_log.txt"

# 기존에 떠 있던 프로세스가 있으면 정리
subprocess.run(["pkill", "-f", "streamlit run app.py"], stderr=subprocess.DEVNULL)

process = subprocess.Popen(
    [
        "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.enableCORS", "false",
        "--server.enableXsrfProtection", "false",
        "--server.enableWebsocketCompression", "false",
    ],
    stdout=open(LOG_PATH, "w"),
    stderr=subprocess.STDOUT,
)

booted = False
for _ in range(30):
    time.sleep(1)
    log_text = open(LOG_PATH).read()
    if "You can now view your Streamlit app" in log_text or "Local URL" in log_text:
        booted = True
        break

if not booted:
    print("Streamlit이 기동되지 않았습니다. 아래 로그를 확인하세요.\n")
    print(log_text)
else:
    from google.colab import output
    print("Streamlit 정상 기동. 아래에서 새 창으로 대시보드를 엽니다.")
    output.serve_kernel_port_as_window(8501)

    # 새 창 팝업이 막혀 있다면 아래 줄의 주석을 풀어 노트북 내부에 바로 임베드할 수 있습니다.
    # output.serve_kernel_port_as_iframe(8501, height=900)
